# Importation et définition de variables globales

In [1]:
import gymnasium as gym
import numpy as np
import torch
import random
import ale_py
import gc

from agent.rainbow_agent import DQNAgent as Rainbow

from agent.no_noisy_no_categ_agent import DQNAgent as NoNoisyNoCateg
from reseaux.nn_no_noisy_no_categ_no_duel import Network as NnNoThree
from reseaux.nn_no_noisy_no_categ import Network as NnDueling

from agent.no_noisy_agent import DQNAgent as NoNoisy
from reseaux.nn_no_noisy_no_duel import Network as NnCateg
from reseaux.nn_no_noisy import Network as NnNoNoisy

from utils.processing import show_latest_video

In [2]:
# Environment
env = gym.make("ALE/Freeway-v5", render_mode="rgb_array")

# Set random seed

In [3]:
seed = 777

def seed_torch(seed):
    torch.manual_seed(seed)
    if torch.backends.cudnn.enabled:
        torch.cuda.manual_seed(seed)
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True

np.random.seed(seed)
random.seed(seed)
seed_torch(seed)

# Initialisation

In [4]:
# paramètres
num_frames = 20
memory_size = 100000 #ou 10 000 d'après le code original
batch_size = 128
target_update = 100
epsilon_decay = 1 / 2000

# Agent

In [ ]:
score_rainbow = np.zeros(num_frames)

for iter in range(num_frames): 
    print(iter)
    # Définition
    agent_rainbow = Rainbow(env, memory_size, batch_size, target_update, seed)

    # Entraînement
    agent_rainbow.train(num_frames)

    # Test
    score_rainbow[iter] = agent_rainbow.test()

    # Nettoyage mémoire
    agent_rainbow.cleanup()
    del agent_rainbow
    gc.collect()

In [6]:
score_rainbow

array([ 0.0000e+00,  0.0000e+00, -7.7000e+00,  0.0000e+00,  4.1024e+02,
       -1.3500e+00,  0.0000e+00,  1.0450e+03,  0.0000e+00,  0.0000e+00,
        0.0000e+00,  0.0000e+00, -2.0480e+01, -1.9970e+01, -2.0000e-01,
        5.0000e-01, -2.0480e+01,  1.0450e+03,  1.0450e+03, -2.0160e+01])

In [ ]:
score_rainbow = np.zeros(num_frames)
score_no_three = np.zeros(num_frames)
score_no_noisy_no_categ = np.zeros(num_frames)
score_no_noisy_no_duel = np.zeros(num_frames)
score_no_noisy = np.zeros(num_frames)

for iter in range(num_frames): 
    # Définition
    agent_rainbow = Rainbow(env, memory_size, batch_size, target_update, seed)
    agent_no_three = NoNoisyNoCateg(NnNoThree, env, memory_size, batch_size, target_update, epsilon_decay, seed)
    agent_no_noisy_no_categ = NoNoisyNoCateg(NnDueling, env, memory_size, batch_size, target_update, epsilon_decay, seed)
    agent_no_noisy_no_duel = NoNoisy(NnCateg, env, memory_size, batch_size, target_update,  epsilon_decay, seed)
    agent_no_noisy = NoNoisy(NnNoNoisy, env, memory_size, batch_size, target_update, epsilon_decay, seed)
    
    # Entraînement
    agent_rainbow.train(num_frames)
    agent_no_three.train(num_frames)
    agent_no_noisy_no_categ.train(num_frames)
    agent_no_noisy_no_duel.train(num_frames)
    agent_no_noisy.train(num_frames)

    # Test
    agent_rainbow.test()
    agent_no_three.test()
    agent_no_noisy_no_categ.test()
    agent_no_noisy_no_duel.test()
    agent_no_noisy.test()

    # Nettoyage mémoire
    agent_rainbow.cleanup()
    del agent_rainbow
    gc.collect()